# Reranking and Context Selection
Companion notebook for the **Reranking and Context Selection** section of Chapter 10.

First-stage retrieval is fast but only "good enough." A reranker gives a second, more careful pass. The goal isn't to pass more context to the model -- it's to pass the **best** context.

A small deterministic reranker (word-overlap between query and chunk) standing in for a real cross-encoder model:

In [1]:
class WordOverlapReranker:
    def score(self, query: str, text: str) -> float:
        query_words = set(query.lower().split())
        text_words = set(text.lower().split())
        if not query_words:
            return 0.0
        return len(query_words & text_words) / len(query_words)

This is the book's `select_context`, unchanged:

In [2]:
def select_context(query, candidates, reranker, top_k=5, min_score=0.2):
    scored = []
    for chunk in candidates:
        score = reranker.score(query, chunk["text"])
        if score >= min_score:
            scored.append({**chunk, "rerank_score": score})
    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]

In [3]:
candidates = [
    {"chunk_id": "c1", "text": "Refunds are allowed within 30 days of purchase for annual plans."},
    {"chunk_id": "c2", "text": "Enterprise plans follow a separate 45 day refund window."},
    {"chunk_id": "c3", "text": "Our support hours are 9am to 6pm on weekdays."},
    {"chunk_id": "c4", "text": "Partial refunds are not offered for partially used subscription periods."},
    {"chunk_id": "c5", "text": "The mobile app requires iOS 15 or later."},
]

selected = select_context(
    "What is the refund window for annual plans?",
    candidates,
    reranker=WordOverlapReranker(),
    top_k=3,
    min_score=0.2,
)
for c in selected:
    print(f"{c['chunk_id']}  score={c['rerank_score']:.2f}  {c['text']}")

c1  score=0.25  Refunds are allowed within 30 days of purchase for annual plans.


`c5` (unrelated to refunds at all) never makes it past `min_score`; `c3` (support hours) either scores too low or loses out to more relevant chunks once `top_k` trims the list. In practice you'd retrieve 20-50 candidates, rerank all of them, and keep only a handful -- cheaper and cleaner for the model than passing everything through.

A **score-gap rule** is a useful addition: if scores fall off sharply after the top few, stop there instead of padding the prompt with weak evidence:

In [4]:
def select_context_with_gap_rule(query, candidates, reranker, top_k=5, min_score=0.2, gap_ratio=0.5):
    scored = select_context(query, candidates, reranker, top_k=len(candidates), min_score=min_score)
    kept = scored[:1] if scored else []
    for prev, cur in zip(scored, scored[1:]):
        if cur["rerank_score"] < prev["rerank_score"] * gap_ratio:
            break
        kept.append(cur)
    return kept[:top_k]


selected_gapped = select_context_with_gap_rule(
    "What is the refund window for annual plans?", candidates, reranker=WordOverlapReranker()
)
print("kept after score-gap rule:", [c["chunk_id"] for c in selected_gapped])

kept after score-gap rule: ['c1']
